# Production plan with capacity options

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/capacity_options.ipynb)

A company assembles electric bicycles in three plants and plans how many bikes each plant assembles each month from January to June. In the fifth version of the model, M4, the orders come from six retailers at the end of a distribution network in Spain, and part of the demand that a retailer does not receive in its month is lost. Under forecast 5, M4 has the three plants at their capacity from March and the storage of the network full before the peak, and 140 orders are lost.

This notebook uses M4B, a version built on M4. With it, whoever plans can see how far the orders can grow with every order delivered, how much a price promotion must raise the orders for the margin (defined in §2) to reach that of the plan without promotion, and what closing plant 3 saves and costs. The plans start from one that delivers every order, so that each change shows when orders start to be lost and what it costs. M4B therefore starts from a lower demand, the base forecast, given in the first table: an invented demand, between 77,8 % and 81,8 % of forecast 5 in each month and 79 % in the six months. Under it, with plant 3 open, every order is delivered and plant 3 has spare capacity. The orders of each month can grow over the base forecast, and the price of a month can fall with a promotion. The plan has two capacity options, each with its cost: overtime at plants 1 and 2, and rented storage next to the central warehouse. Plant 3, the most expensive plant, can be closed for the six months, and while it is open it has a fixed cost. Each plan is made for one demand, as if it were known.

Contents of the notebook:

1. Base forecast, orders of each retailer and capacity options.
2. Mixed-integer linear model with capacity options.
3. Plan of a scenario, built with the form: demand growth, price promotion, plant 3, overtime and rented storage.

The cells run in order, with ▶ or *Shift + Enter*. The form of the last cell (sliders, tick boxes and a drop-down list) builds the scenario: after changing a value, run the cell again. Outside Colab the form is a set of lines such as `demand_growth = 0`: edit the value and run the cell.

The table gives the symbols that M4B adds to those of M0 to M4, which the [document of M4](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/retailers.md) lists.

| Symbol | Meaning | Term | Unit |
|---|---|---|---|
| $D_t$ | Orders of the six retailers together in month $t$ before any growth | base forecast | bikes/month |
| $\sigma_r$ | Share of the orders of each month that retailer $r$ places: the shares of M4, from 10 % to 22 %, which add up to 100 % | share | fraction (0,14 for 14 %) |
| $\gamma_t$ | Change of the orders of every retailer in month $t$ over the base forecast, as a fraction (0,10 for 10 %); a negative value lowers them | demand growth | — |
| $\pi_t$ | Price that a retailer pays for a bike ordered in month $t$: 700 €, times one minus the discount (a percentage, chosen in the form) in the months of a promotion; it is also the lost-sale cost of that month, and it replaces the lost-sale cost $c^l_r$ of M4 | price | €/bike |
| $\bar o_f$ | Bikes that plant $f$ can assemble in a month above its assembly capacity; $\bar o_3 = 0$ | overtime limit | bikes/month |
| $c^o$ | Cost of a bike assembled in overtime above the unit assembly cost of its plant | overtime cost | €/bike |
| $\bar u$ | Bikes that can be held next to the central warehouse above its storage capacity | rented storage limit | bikes |
| $c^u$ | Cost of holding one bike in rented storage for one month, above the holding cost of the central warehouse | rented storage cost | €/(bike·month) |
| $c^F_f$ | Cost of keeping plant $f$ open for one month, whatever it assembles; $c^F_1 = c^F_2 = 0$ | fixed cost | €/month |
| $o_{ft}$ | Bikes that plant $f$ assembles in overtime in month $t$, of the $p_{ft}$ it assembles (§2) | overtime | bikes/month |
| $u_t$ | Bikes held in rented storage at the end of month $t$, of the $h_{wt}$ held at the central warehouse, $w$ (§2) | rented storage | bikes |
| $\delta_f$ | 1 if plant $f$ is open from January to June, 0 if it is closed; $\delta_1 = \delta_2 = 1$ | — | binary |

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/capacity_options.md) explains all the data and results. The places and distances are real; the other data are invented.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import capacity_options as co

## 1. Base forecast, orders of each retailer and capacity options

The first table gives the base forecast $D_t$, the orders of the six retailers together in each month; the orders of each retailer, $\sigma_r D_t$; and the price of a bike in each month without a promotion. The second gives the capacity options with their cost, and the fixed cost of plant 3. The rest of the data are those of M4.

In [ ]:
# @title Base forecast, orders and capacity options { display-mode: "form" }
co.show_data()

## 2. Mixed-integer linear model with capacity options

$$
\begin{aligned}
\min \quad & \sum_{t \in T} \Big( \sum_{f \in F} \left( c_f\, p_{ft} + c^o\, o_{ft} \right) + \sum_{i \in V} c^h_i\, h_{it} + c^u\, u_t + \sum_{a \in A} c^x_a\, x_{at} + \sum_{r \in R} \left( c^b_r\, b_{rt} + \pi_t\, l_{rt} \right) \Big) + \sum_{f \in F} |T|\, c^F_f\, \delta_f \\
\text{s.t.} \quad & h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} = \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} + h_{it} && \forall i \in V,\ t \in T && \text{(stock balance at each node)} \\
& \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} \le h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} && \forall i \in V,\ t \in T && \text{(shipment limit at each node)} \\
& b_{r,t-1} + d_{rt} = s_{rt} + b_{rt} + l_{rt} && \forall r \in R,\ t \in T && \text{(backorder balance)} \\
& b_{rt} = \beta_r \left( b_{r,t-1} + d_{rt} - s_{rt} \right) && \forall r \in R,\ t < \tau && \text{(backorder share)} \\
& x_{at} \le K^x_a && \forall a \in A,\ t \in T && \text{(arc capacity)} \\
& \sum_{a \in A^{\text{in}}_i} x_{at} \le K^n_i && \forall i \text{ with a node capacity},\ t \in T && \text{(node capacity)} \\
& h_{it} \le K^h_i && \forall i \in V \setminus \{w\},\ t \in T && \text{(storage capacity)} \\
& h_{wt} - u_t \le K^h_w && \forall t \in T && \text{(storage capacity of the central warehouse, relaxed by the rented storage)} \\
& u_t \le \bar u && \forall t \in T && \text{(rented storage)} \\
& p_{ft} - o_{ft} \le K_{ft}\, \delta_f && \forall f \in F,\ t \in T && \text{(assembly capacity, relaxed by the overtime)} \\
& o_{ft} \le \bar o_f\, \delta_f && \forall f \in F,\ t \in T && \text{(overtime)} \\
& b_{r\tau} = 0 \quad \forall r \in R, \qquad h_{i\tau} \ge h^{\text{end}}_i \quad \forall i \in V && && \text{(backorders and required final stock at the end)} \\
& p_{ft},\ o_{ft},\ h_{it},\ u_t,\ x_{at},\ s_{rt},\ b_{rt},\ l_{rt} \ge 0, \qquad \delta_f \in \{0; 1\}
\end{aligned}
$$

with $h_{i0}$ given, $b_{r0} = 0$, $w$ the central warehouse and $|T| = 6$ the number of months. The orders are $d_{rt} = \sigma_r\, D_t\, (1 + \gamma_t)$. In the optimal plan the overtime $o_{ft}$ is part of what the plant assembles, $p_{ft}$, and the rented storage $u_t$ is part of what the central warehouse holds, $h_{wt}$. The model does not impose it, but the overtime and the rented storage have a positive cost, so a plan of least cost never has more overtime than the plant assembles, or more rented storage than the warehouse holds. A closed plant assembles nothing: with $\delta_f = 0$ the assembly capacity and the overtime are 0.

The margin of a plan is the price of all the orders, delivered or not, less the cost of the plan. An order lost in the month in which it is placed counts its price in the orders and, as a lost sale, the same price in the cost: the two cancel, as if it had not been placed (assumption 4 below gives the other case). The price of the orders is a datum, so the plan of least cost is the plan of highest margin. A cost that is paid in every plan does not change which plan is optimal: it only lowers every margin by the same amount. Plants 1 and 2 are always open (assumption 5 below), so a fixed cost of theirs would be such a cost, and it is left out. Whoever plans fixes $\delta_3$ as a datum, 1 if plant 3 is open and 0 if it is closed, or leaves it as a binary variable: then the model compares the cost of the whole plan with plant 3 open, fixed cost included, and with plant 3 closed. $\delta_3$ has to be binary because the fixed cost is paid whole or not at all: with $\delta_3$ between 0 and 1, the constraints $p_{ft} - o_{ft} \le K_{ft}\, \delta_f$ and $o_{ft} \le \bar o_f\, \delta_f$ would let the plan buy a fraction of the capacity and of the overtime of plant 3 for the same fraction of its fixed cost. With $\delta_3$ as a binary variable among continuous ones, the model is a mixed-integer linear model; with $\delta_3$ as a datum, it is linear.

Against M4, the orders are $\sigma_r D_t (1 + \gamma_t)$ instead of data, the lost-sale cost of each month is its price $\pi_t$ instead of $c^l_r$, and the objective adds the cost of the overtime, of the rented storage and of plant 3 while it is open. The assembly capacity is relaxed by the overtime and the storage capacity of the central warehouse by the rented storage, and two new constraints limit them: at most $\bar o_f$ bikes of overtime a month at each plant, and at most $\bar u$ bikes in rented storage.

The assumptions of M4 hold, with three changes:

- Instead of assumption 2 of M4: costs are proportional to the quantities, as in M4, except the fixed cost of plant 3 while it is open.
- Instead of assumption 6 of M4: each retailer orders its share $\sigma_r$ of the base forecast raised by the demand growth, not of forecast 5.
- Instead of assumption 9 of M4: the lost-sale cost, the same at every retailer as in M4, is the price of the month.

Fractional quantities stay allowed, and with a demand growth the orders and the plan are in general not whole numbers. M4B adds six assumptions of its own:

1. Each demand is planned as if it were known.
2. The growth of the orders of a promotion is a datum chosen by whoever plans, not a response that the model represents, and the promotion takes no orders from the other months.
3. The discount applies to every order of the months of the promotion, and the promotion has no other cost, such as advertising.
4. An order backordered in one month and lost in a later month costs the price of that later month: the model counts the backorders of a retailer together, without the month in which each was placed. Compared with an order that is not placed, an order of a month of a promotion that is lost in a later month at full price takes the discount from the margin, and an order lost in a month of a promotion adds the discount to it. An order lost in its own month leaves the margin as if it had not been placed.
5. Plants 1 and 2 are open in all six months, and plant 3 is open or closed for all six months.
6. Overtime is available only at plants 1 and 2, and rented storage only next to the central warehouse.

## 3. Plan of a scenario

A scenario is a set of values of the form: the demand of each month, its price, the setting of plant 3 and the capacity options available. The cell solves M4B for the scenario and gives the cost and the margin of the plan; the margin under the base forecast, with the same values of the form except the demand growth and the promotion, to which the margin of the scenario is compared; the capacity utilization of each plant, one figure for the six months together; the price and the orders of each month; the bikes assembled at each plant, with the part assembled in overtime; the seasonal stock at the end of each month by stage, with the part held in rented storage; and the lost sales of each retailer. As in M2 to M4, there is no safety stock (assumption 6 of M2), so the seasonal stock is all the stock held, the required final stock included. A stage is a group of nodes: the plants, the central warehouse, the regional warehouses and the local warehouses; the rail terminal and the retailers hold no stock. The capacity utilization of a plant is what it assembles in the six months, overtime included, as a share of its assembly capacity of the six months; with overtime it can exceed 100 %, and for a closed plant it is 0 %. The figure gives it month by month: what the plant assembles in the month over its capacity of the month. The figure has two panels: on the left, the capacity utilization of each plant in each month, in %, with a dashed line at 100 %, the assembly capacity; on the right, the lost sales of each retailer in each month, in bikes. If several plans have the optimal cost, the cell says so.

**Demand growth.** `demand_growth` raises the orders of every retailer in every month by the same percentage over the base forecast; a negative value lowers them. With plant 3 open and no capacity option, raise it from 0 to 40 %: from which growth on are orders lost, and what is the capacity utilization of plant 3, for the six months together, when that happens?

**Price promotion.** The tick boxes choose the months of the promotion, `discount` lowers the price of those months, and `growth_of_their_orders` is the growth of their orders, which replaces `demand_growth` in those months. With plant 3 open, no capacity option and `demand_growth` at 0, put a discount of 5 % in April and May and raise the growth of their orders until the margin of the scenario reaches the margin under the base forecast; then untick April and May, tick January, February and March, and do the same. Repeat both with a discount of 7,5 %: with that discount, do April and May need more growth than January to March, and why? With a discount of 7,5 % and the growth that reaches the margin under the base forecast, which orders are lost in the promotion of April and May?

**Plant 3.** With `plant_3` set to `open` or `closed`, plant 3 is open or closed for the six months, as a datum. With `variable`, $\delta_3$ is a binary variable (§2). `fixed_cost_of_plant_3` is paid for each of the six months if plant 3 is open. Its default is 0 €/month, so that the cost of a plan with plant 3 open leaves out the fixed cost: with plant 3 open as a datum it is the same in every plan and does not change which plan is optimal (§2). The case gives 20.000 €/month (second table of §1). Set it to 20.000 €/month in two cases: to compare a plan with plant 3 open with one with plant 3 closed, which has no fixed cost; and with `variable`, because then the fixed cost is compared with what closing plant 3 adds to the rest of the cost of the plan. Close plant 3 under the base forecast (`demand_growth` at 0): at which retailers are orders lost, and why at those? Is the margin of the plan with plant 3 closed above that of the plan with plant 3 open and a fixed cost of 20.000 €/month, both under the base forecast? With `variable` and a fixed cost of 20.000 €/month, is plant 3 open or closed in the optimal plan under the base forecast, and from which demand growth on, raising it from 0, does that change?

**Overtime and rented storage.** `overtime` lets plants 1 and 2 assemble above their capacity, up to the limits of the second table of §1; `rented_storage` lets the central warehouse hold up to 200 bikes above its storage capacity. The default costs are those of that table. With plant 3 closed and `demand_growth` at 0, compare the two options one at a time and both together: with which one is every order delivered, which one lowers the lost sales and the cost least, and what do both together add to overtime alone?

In [ ]:
# @title Plan of a scenario { display-mode: "form" }
# Demand growth over the base forecast, in every month (%)
demand_growth = 0  # @param {type:"slider", min:-20, max:50, step:1}
# Months of the price promotion
january = False  # @param {type:"boolean"}
february = False  # @param {type:"boolean"}
march = False  # @param {type:"boolean"}
april = False  # @param {type:"boolean"}
may = False  # @param {type:"boolean"}
june = False  # @param {type:"boolean"}
# Discount on the price of those months (%) and growth of their orders (%)
discount = 0  # @param {type:"slider", min:0, max:30, step:0.5}
growth_of_their_orders = 0  # @param {type:"slider", min:0, max:150, step:1}
# Plant 3 open, closed or a binary variable, and what it costs to keep it open (EUR/month)
plant_3 = "open"  # @param ["open", "closed", "variable"]
fixed_cost_of_plant_3 = 0  # @param {type:"slider", min:0, max:50000, step:1000}
# Overtime at plants 1 and 2 (EUR/bike) and rented storage (EUR/(bike*month))
overtime = False  # @param {type:"boolean"}
overtime_cost = 100  # @param {type:"slider", min:10, max:200, step:10}
rented_storage = False  # @param {type:"boolean"}
rented_storage_cost = 12  # @param {type:"slider", min:1, max:40, step:1}
co.plan(growth=demand_growth,
        promotion_months=co.chosen_months(january, february, march, april, may, june),
        discount=discount, promotion_growth=growth_of_their_orders,
        plant_3=plant_3, fixed_cost=fixed_cost_of_plant_3,
        overtime=overtime, overtime_cost=overtime_cost,
        rented_storage=rented_storage, rented_cost=rented_storage_cost);

The script `capacity_options_solver.py`, in the folder of the case, prints every table of the [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/capacity_options.md) except the notation.